In [1]:
from modelscope import AutoModelForCausalLM

model_name = "openai-community/gpt2"
# gpt_model=AutoModel.from_pretrained(model_name)
gpt_model = AutoModelForCausalLM.from_pretrained(model_name)
print(gpt_model)


C:\Users\17246\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [2]:
from modelscope import AutoTokenizer

gpt_tokenizer = AutoTokenizer.from_pretrained(model_name)

gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

gpt_tokenizer.pad_token_id


50256

In [16]:
from modelscope import MsDataset

dataset = MsDataset.load('modelscope/chinese-poetry-collection', subset_name='default', split='train')

data = dataset.to_hf_dataset().select(range(20))

data[:2]

2025-08-08 19:05:28,897 - modelscope - WARNING - Use trust_remote_code=True. Will invoke codes from chinese-poetry-collection. Please make sure that you can trust the external codes.
2025-08-08 19:05:30,580 - modelscope - WARNING - Reusing dataset dataset_builder (C:\Users\17246\.cache\modelscope\hub\datasets\modelscope\chinese-poetry-collection\master\data_files)
2025-08-08 19:05:30,581 - modelscope - INFO - Generating dataset dataset_builder (C:\Users\17246\.cache\modelscope\hub\datasets\modelscope\chinese-poetry-collection\master\data_files)
2025-08-08 19:05:30,583 - modelscope - INFO - Reusing cached meta-data file: C:\Users\17246\.cache\modelscope\hub\datasets\modelscope\chinese-poetry-collection\master\data_files\7c9a7977d937face2055b6145eaf516f


{'text1': ['半生长以客为家，罢直初来瀚海槎。始信人间行不尽，天涯更复有天涯。',
  '南州未识异州苹，初向沙头问水神。料得行藏无用卜，乘桴人是北来人。']}

In [9]:
from torch.utils.data import Dataset, DataLoader


class Dataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.encodings = []

        for poem in data['text1']:
            poem = poem + gpt_tokenizer.eos_token
            encoded = tokenizer(
                poem,
                max_length=max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            input_ids = encoded['input_ids'].squeeze()
            self.encodings.append(input_ids)

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        input_ids = self.encodings[idx]
        labels = input_ids.clone()
        labels[:-1] = input_ids[1:]
        labels[-1] = -100
        return input_ids, labels


dataset = Dataset(data, gpt_tokenizer)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

for input_ids, labels in dataloader:
    print(input_ids)
    print(labels)
    break

tensor([[33699,   123,   162,   223,   102, 39355,   225, 34932,   234, 49035,
           118,   162,   109,   253, 20046,    94,   171,   120,   234,   164,
           121,   105, 43889,   228, 49011, 17739,   111, 34402,   241,   164,
           115,   107,   165,   243,   123, 16764,   165,   119,   226, 20998,
           114, 37239,   122, 28156,   222,   161,   109,   109, 30585,   224,
         37239,   228,   171,   120,   234,   164,   100,   223, 21689, 49035,
           104, 37239,   223, 33232,   228,   162,   101,   103,   161,    94,
           246, 16764, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 5

In [13]:
from torch import optim, nn

optimizer = optim.Adam(gpt_model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

for _ in range(20):
    for input_ids, labels in dataloader:
        optimizer.zero_grad()
        output = gpt_model(input_ids, labels=input_ids)
        loss = output.loss
        loss.backward()
        optimizer.step()
    print(loss.item())

1.1217950582504272
1.0904548168182373
0.4923807978630066
0.1209651380777359
0.14724603295326233
0.06303150951862335
0.081058070063591
0.011465790681540966
0.037908948957920074
0.07031681388616562
0.03956277295947075
0.0030427230522036552
0.0019523577066138387
0.025461973622441292
0.03859936073422432
0.014493527822196484
0.01771090365946293
0.0011944613652303815
0.013406096026301384
0.013160471804440022


In [17]:
def generate_poem(sentence, max_length=128):
    input_ids = gpt_tokenizer.encode(sentence, return_tensors='pt')
    output = gpt_model.generate(input_ids, max_length=max_length, do_sample=True,
                                pad_token_id=gpt_tokenizer.pad_token_id)
    poem = gpt_tokenizer.decode(output[0], skip_special_tokens=True)
    return poem
print(generate_poem('南'))

南州未识异州苹，初向沙头问水神。料得行藏无用卜，乘桴人是北来人。
